<a href="https://colab.research.google.com/github/azraisik/yapay-sinir-aglari/blob/kerem%2Ftasar%C4%B1m-mimari/TASARIM_VE_MIMARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Adım: Kütüphaneleri projemize dahil ediyoruz
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2. Adım: Temizlenmiş veri setimizi içeri aktarıyoruz
df = pd.read_csv("SLA_Temiz_Veri.csv")

# Verimizin ilk 5 satırına göz atıp doğru yükledik mi diye kontrol edelim
print("--- Veri Setinin İlk 5 Satırı ---")
print(df.head())

--- Veri Setinin İlk 5 Satırı ---
   Case ID    Variant Priority         Issue Type Report Channel  Step_count  \
0  INC0001  Variant 4   Medium  Performance Issue        Website           7   
1  INC0002  Variant 3     High  Performance Issue            App           8   
2  INC0003  Variant 9     High    Feature Request        Website          10   
3  INC0004  Variant 1   Medium           Incident          Email           6   
4  INC0005  Variant 3      Low  Performance Issue        Website           8   

   Reassignment_count  Escalation_count  Has_Bounce  Workload_Index  \
0                   5               1.0           0              35   
1                   5               1.0           0              40   
2                   5               1.0           0              50   
3                   3               0.0           0              18   
4                   6               1.0           0              48   

   Open_Hour  Is_Weekend  SLA_Violation  
0         11    

In [2]:
# 'Case ID' sütunu tahminde bir işe yaramayacağı için onu tablodan tamamen siliyoruz
df = df.drop(columns=['Case ID'])

print("\n--- Case ID Silindikten Sonra Kalan Sütunlar ---")
print(df.columns)


--- Case ID Silindikten Sonra Kalan Sütunlar ---
Index(['Variant', 'Priority', 'Issue Type', 'Report Channel', 'Step_count',
       'Reassignment_count', 'Escalation_count', 'Has_Bounce',
       'Workload_Index', 'Open_Hour', 'Is_Weekend', 'SLA_Violation'],
      dtype='object')


In [3]:
# Dönüştürülecek kategorik (metin içeren) sütunların listesi
kategorik_sutunlar = ['Variant', 'Priority', 'Issue Type', 'Report Channel']

# get_dummies komutuyla bu sütunları 0 ve 1'lere dönüştürüyoruz
# drop_first=True yapıyoruz ki matematiksel olarak birbirini tekrar eden gereksiz sütunlar oluşmasın
df_hazir = pd.get_dummies(df, columns=kategorik_sutunlar, drop_first=True)

# Yapay zekaya girecek Özellikleri (X) ve tahmin edilmek istenen Hedefi (y) ayıralım
# Hedefimiz: SLA_Violation (İhlal var mı=1, yok mu=0)
X = df_hazir.drop(columns=['SLA_Violation']).astype(np.float32)
y = df_hazir['SLA_Violation'].values.astype(np.float32)

print(f"\nGiriş verisinin (X) yeni boyutu (Satır, Sütun): {X.shape}")


Giriş verisinin (X) yeni boyutu (Satır, Sütun): (30618, 30)


In [4]:
# 1. Veriyi %80 Eğitim, %20 Test olarak bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Sayıları birbirine eşitlemek için ölçeklendiriciyi (StandardScaler) hazırlıyoruz
scaler = StandardScaler()

# Sadece eğitim verisine bakarak kuralları öğren (fit) ve veriyi dönüştür (transform)
X_train_scaled = scaler.fit_transform(X_train)

# Test verisini ise eğitimden öğrendiğin kurallara göre dönüştür (asla test verisine bakarak fit yapma!)
X_test_scaled = scaler.transform(X_test)

print("Eğitim setindeki örnek sayısı:", X_train_scaled.shape[0])
print("Test setindeki örnek sayısı:", X_test_scaled.shape[0])
print("Giriş yapacak toplam özellik (sütun) sayısı:", X_train_scaled.shape[1])

Eğitim setindeki örnek sayısı: 24494
Test setindeki örnek sayısı: 6124
Giriş yapacak toplam özellik (sütun) sayısı: 30


In [5]:
# Sıralı bir model zinciri oluşturuyoruz
model = Sequential()

# 1. GİZLİ KATMAN: 64 nöronlu, aktivasyon fonksiyonu ReLU.
# input_shape ile içeri girecek sütun sayısını (özellik sayısını) modele söylüyoruz.
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
# DROPOUT: Eğitim sırasında nöronların %20'sini rastgele kapat ki ezber yapmasın.
model.add(Dropout(0.2))

# 2. GİZLİ KATMAN: 32 nöronlu, yine ReLU aktivasyonlu.
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

# ÇIKIŞ KATMANI: Sadece 1 nöron (Çünkü cevap tek: İhlal var mı=1, yok mu=0)
# Aktivasyon Sigmoid: Bize 0 ile 1 arasında bir olasılık (örneğin 0.78 yani %78 ihlal riski) üretecek.
model.add(Dense(1, activation='sigmoid'))

# MODELİ DERLEME: Modelin çalışma kurallarını belirliyoruz.
model.compile(
    optimizer='adam',                   # Ağırlıkları güncelleyen akıllı şoförümüz
    loss='binary_crossentropy',         # İkili sınıflandırma hata ölçerimiz
    metrics=['accuracy']                # Başarıyı 'doğruluk' (Accuracy) ile takip et
)

# Modelin özet tablosunu ekrana bastıralım
print("\n--- YAPAY SİNİR AĞI MİMARİ ÖZETİ ---")
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- YAPAY SİNİR AĞI MİMARİ ÖZETİ ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
print("\n--- MODEL EĞİTİMİ BAŞLIYOR ---")

# Modeli eğitiyoruz ve eğitim geçmişini 'history' değişkenine kaydediyoruz
history = model.fit(
    X_train_scaled, y_train,
    epochs=20,                  # Tüm verilerin üzerinden 20 kere geçerek öğrenecek
    batch_size=32,              # Verileri 32'şerli paketler halinde sisteme sokacak
    validation_data=(X_test_scaled, y_test), # Her turda kenara ayırdığımız test verisiyle kendini test edecek
    verbose=1                   # Eğitim sürecini ekranda satır satır göster
)


--- MODEL EĞİTİMİ BAŞLIYOR ---
Epoch 1/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9626 - loss: 0.1161 - val_accuracy: 0.9815 - val_loss: 0.0600
Epoch 2/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9791 - loss: 0.0651 - val_accuracy: 0.9827 - val_loss: 0.0562
Epoch 3/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9802 - loss: 0.0599 - val_accuracy: 0.9827 - val_loss: 0.0561
Epoch 4/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9809 - loss: 0.0581 - val_accuracy: 0.9815 - val_loss: 0.0570
Epoch 5/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9811 - loss: 0.0570 - val_accuracy: 0.9827 - val_loss: 0.0552
Epoch 6/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9812 - loss: 0.0566 - val_accuracy: 0.9825 - val_loss: 0.0560
Epoch 7/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9816 - loss: 0.0556 - val_accuracy: 0.9827 - val_loss: 0.0553
Epoch 8/20
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9813